<img src="../../branding/aa-wordmark.png" alt="AA Engineering Toolkit" width="320px">

*Areeb Ahmed | Mechanical Engineer*

# PyBaMM: physics-based cell models with degradation

PyBaMM (Sulzer et al. 2021) solves electrochemical models such as the single particle model with electrolyte
(SPMe) and the Doyle-Fuller-Newman model (DFN). It needs compiled solvers, so it is an optional extra that does
not run in the browser:

```bash
pip install -e ".[battery]"
```

Without PyBaMM the PyBaMM cells are skipped (`HAVE_PYBAMM = False`) and the committed reference grid is shown instead.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from engtoolkit.catalogs import get_cell
from engtoolkit.core.plotting import use_style
from engtoolkit.electrical.battery import Cooling, ECMParams, ThermalModel, pybamm_models as pm, reference, simulate

use_style()
HAVE_PYBAMM = pm.available()
cell = get_cell("LG M50 21700 (NMC811)")
print("PyBaMM available:", HAVE_PYBAMM)

## DFN discharge vs the equivalent-circuit model

Both use the catalog cell's thermal mass (m·cp) and surface area, with h = 10 W/(m²·K). Dashed: DFN, solid: ECM.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ecm = ECMParams.from_cell(cell)
for c_rate in (0.5, 1.0, 2.0):
    if HAVE_PYBAMM:
        dfn = pm.discharge(c_rate, 298.15, h=10.0, heat_capacity=cell.mass * cell.cp, area=cell.surface_area)
    else:
        dfn = reference.discharge(c_rate, 298.15)
    th = ThermalModel.for_cell(cell, Cooling("fixed_h", T_coolant=298.15, h_fixed=10.0))
    sim = simulate(ecm, "discharge_cc", c_rate=c_rate, thermal=th)
    charge = np.concatenate([[0.0], np.cumsum(np.diff(sim.t) * sim.current[:-1])]) / 3600
    line, = ax1.plot(dfn["capacity_Ah"], dfn["voltage"], "--", label=f"DFN {c_rate:g}C")
    ax1.plot(charge, sim.voltage, color=line.get_color(), label=f"ECM {c_rate:g}C")
    ax2.plot(np.asarray(dfn["time"]) / 60, np.asarray(dfn["temperature"]) - 273.15, "--", color=line.get_color())
    ax2.plot(sim.t / 60, sim.temperature - 273.15, color=line.get_color())
ax1.set(xlabel="Charge [Ah]", ylabel="Voltage [V]")
ax2.set(xlabel="Time [min]", ylabel="Temperature [°C]")
ax1.legend(fontsize=8);

## Fitting the equivalent circuit to the DFN

A 1C pulse from 60 % SOC; the overpotential (bulk OCV minus terminal voltage) is fitted with two RC pairs.
These are the values stored in `catalogs/data/cells.yaml`.

In [ ]:
if HAVE_PYBAMM:
    for T in (283.15, 298.15, 313.15):
        f = pm.fit_ecm(T)
        print(f"{T - 273.15:4.0f} °C: R0 = {f['R0']*1e3:.2f} mΩ, R1 = {f['R1']*1e3:.2f} mΩ (τ1 = {f['tau1']:.1f} s), "
              f"R2 = {f['R2']*1e3:.2f} mΩ (τ2 = {f['tau2']:.0f} s), rms = {f['rms_V']*1e3:.2f} mV")
else:
    print(reference.load()["ecm_fit"])

## Degradation: SEI growth and lithium plating

OKane2022 parameters with solvent-diffusion-limited SEI and partially reversible plating. A short run here
(20 cycles); the committed grid has 500 cycles at three C-rates and three temperatures (about 7 minutes to
generate on 4 cores).

In [ ]:
if HAVE_PYBAMM:
    short = pm.degradation(1.0, 298.15, n_cycles=20, save_every=5, heat_capacity=cell.mass * cell.cp,
                           area=cell.surface_area)
    print("relative capacity:", np.round(short["capacity"], 5))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for a in reference.load()["aging"]:
    label = f"{a['c_rate']:g}C, {a['T_ambient'] - 273.15:.0f} °C"
    ax1.plot(a["cycles"], np.asarray(a["capacity"]) * 100, label=label)
    ax2.plot(a["cycles"], a["li_plating"], label=label)
ax1.set(xlabel="Cycle", ylabel="Capacity [%]", title="Capacity (SEI + plating)")
ax2.set(xlabel="Cycle", ylabel="Capacity lost to plating [Ah]", title="Lithium plating (worst when cold)")
ax1.legend(fontsize=7, ncol=3);